In [1]:
!pip install --upgrade onnx
!pip install --upgrade onnx onnxruntime
!pip install --upgrade onnx onnx-simplifier
!pip install --upgrade onnx onnx-simplifier onnxruntime

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com

[notice] A new release of pip is available: 24.0 -> 26.0
[notice] To update, run: python -m pip install --upgrade pip
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com

[notice] A new release of pip is available: 24.0 -> 26.0
[notice] To update, run: python -m pip install --upgrade pip
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com

[notice] A new release of pip is available: 24.0 -> 26.0
[notice] To update, run: python -m pip install --upgrade pip
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com

[notice] A new release of pip is available: 24.0 -> 26.0
[notice] To update, run: python -m pip install --upgrade pip


In [2]:
import os
import random
import numpy as np
import onnx
from onnx import numpy_helper
import onnxruntime
from onnxsim import simplify
from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantType, QuantFormat, quant_pre_process
from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantType, QuantFormat
from onnxruntime.quantization import CalibrationMethod
from PIL import Image
from torchvision import transforms


/usr/local/lib/python3.10/dist-packages/torch/cuda/__init__.py:644: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [3]:

# --- CONFIGURATION ---

BASE_DIR = os.getcwd()
INPUT_MODEL_PATH = os.path.join(BASE_DIR, "weights", "yolop-640-640.onnx")
OUTPUT_DIR = os.path.join(BASE_DIR, "weights")
OUTPUT_MODEL_PATH = os.path.join(OUTPUT_DIR, "yolop-640-640-quantized-int8-Entropy-per_channel-full.onnx")
CALIBRATION_IMG_DIR = os.path.join(BASE_DIR, "dataset", "images", "val")
CALIBRATION_SIZE = 50
TARGET_SIZE = (640, 640)

PRIORITY_IMAGES = ["b1c66a42-6f7d68ca.jpg"] 
HEADS_TO_EXCLUDE = ['det_out'] 
EXCLUSION_DEPTH = 5 

MANUAL_NODES_TO_EXCLUDE = []
NODES_TO_FORCE_INT8 = ["Concat_1534","Expand_1118","Expand_1302","Expand_1426","Expand_686","Expand_810","Expand_994","Mul_1108","Mul_1416","Mul_1416","Mul_800","Reshape_1110","Reshape_1225","Reshape_938","Reshape_1294","Reshape_1418","Reshape_1533","Reshape_630","Reshape_678","Reshape_802","Reshape_917","Reshape_938","Reshape_1246","Reshape_986","ScatterND_1098","ScatterND_1222","ScatterND_1406","ScatterND_1530","ScatterND_790","ScatterND_914","Sigmoid_1279","Sigmoid_971","Sigmoid_663"]

In [4]:


class YOLOPDataReader(CalibrationDataReader):
    """DataReader avec preprocessing YOLOP standard"""
    
    def __init__(self, image_folder, model_path, size_limit=100, priority_list=None):
        self.image_folder = image_folder
        self.enum_data_dicts = None
        self.datasize = 0

        session = onnxruntime.InferenceSession(model_path, providers=['CPUExecutionProvider'])
        self.input_name = session.get_inputs()[0].name

        self.normalize = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

        all_files = [f for f in os.listdir(image_folder) if f.endswith(('.jpg', '.png', '.jpeg'))]

        # Logique de sélection des images
        selected_files = []

        # 1. Ajout des images prioritaires (si elles existent dans le dossier)
        if priority_list:
            for img_name in priority_list:
                if img_name in all_files:
                    selected_files.append(img_name)
                else:
                    print(f" Image prioritaire non trouvée : {img_name}")

        # 2. Gestion de la taille et du complétion aléatoire
        if size_limit is None:
            # Si None, on prend juste la liste prioritaire
            self.image_files = selected_files
        else:
            # Si on a besoin de plus d'images que la liste prioritaire
            remaining_slots = size_limit - len(selected_files)

            if remaining_slots > 0:
                # On filtre pour ne pas reprendre les images déjà sélectionnées
                pool = list(set(all_files) - set(selected_files))
                # On complète aléatoirement
                if len(pool) >= remaining_slots:
                    random_fill = random.sample(pool, remaining_slots)
                    selected_files.extend(random_fill)
                else:
                    selected_files.extend(pool) # On prend tout ce qu'il reste

            self.image_files = selected_files[:size_limit] # Sécurité

        self.datasize = len(self.image_files)
        print(f" DataReader : {self.datasize} images de calibration (Dont {len(priority_list or [])} prioritaires)")


    def letterbox_image(self, image, size):
        iw, ih = image.size
        w, h = size
        scale = min(w/iw, h/ih)
        nw = int(iw * scale)
        nh = int(ih * scale)
        
        image = image.resize((nw, nh), Image.BICUBIC)
        new_image = Image.new('RGB', size, (114, 114, 114))
        new_image.paste(image, ((w - nw) // 2, (h - nh) // 2))
        return new_image
    
    def get_next(self):
        if self.enum_data_dicts is None:
            self.enum_data_dicts = iter(self.load_batch())
        try:
            return next(self.enum_data_dicts)
        except StopIteration:
            return None
    
    def load_batch(self):
        for filename in self.image_files:
            image_path = os.path.join(self.image_folder, filename)
            image = Image.open(image_path).convert("RGB")
            image = self.letterbox_image(image, TARGET_SIZE)
            
            input_tensor = self.normalize(image)
            input_numpy = input_tensor.unsqueeze(0).numpy()
            
            yield {self.input_name: input_numpy}



In [ ]:

def get_bn_weight_names(model_path):
    """Identifie tous les noms de poids BatchNorm dans le modèle"""
    model = onnx.load(model_path)
    bn_weights = []
    
    for initializer in model.graph.initializer:
        if '.bn.weight' in initializer.name or '.bn.bias' in initializer.name:
            bn_weights.append(initializer.name)
    
    print(f" {len(bn_weights)} poids BatchNorm identifiés")
    return bn_weights



In [6]:
def get_sensitive_nodes(model_path, output_names=['det_out'], depth=3, manual_list=[], force_int8_list=[]):
    """
    Remonte le graphe depuis la sortie 'det_out' pour trouver les dernières couches
    (Têtes de détection) et les exclure de la quantification.
    """
    model = onnx.load(model_path)
    nodes_to_exclude = []
    
    output_map = {}
    for node in model.graph.node:
        for out in node.output:
            output_map[out] = node.name
            
    current_nodes = []
    for node in model.graph.node:
        for out_target in output_names:
            if out_target in node.output:
                current_nodes.append(node)
                nodes_to_exclude.append(node.name)

    
    print(f"    • Têtes identifiées pour exclusion : {[n.name for n in current_nodes]}")
 
    if manual_list:
        print(f"    • Traitement de {len(manual_list)} exclusions manuelles...")
        for item in manual_list:
            
            if isinstance(item, int):
                if 0 <= item < len(model.graph.node):
                    target_node = model.graph.node[item]
                    nodes_to_exclude.append(target_node.name)
                    print(f"      - Index {item} -> Nom '{target_node.name}' ajouté.")
                else:
                    print(f"       Index {item} hors limites (Ignoré).")
                    
            elif isinstance(item, str):
                nodes_to_exclude.append(item)
                

    for i in range(depth):
        parents = []
        for node in current_nodes:
            for inp in node.input:
                if inp in output_map:
                    parent_name = output_map[inp]
                    nodes_to_exclude.append(parent_name)
                    
                    for n in model.graph.node:
                        if n.name == parent_name:
                            parents.append(n)
                            break
        current_nodes = parents
    
    nodes_to_exclude = list(set(nodes_to_exclude))
    if force_int8_list:
        print(f"    • Forçage INT8 sur {len(force_int8_list)} noeuds (Retrait de la liste sensible)...")
        original_count = len(nodes_to_exclude)
        nodes_to_exclude = [n for n in nodes_to_exclude if n not in force_int8_list]
        removed_count = original_count - len(nodes_to_exclude)
        print(f"      - {removed_count} noeuds réintégrés à la quantification.")
   

    nodes_to_exclude = sorted(list(set(nodes_to_exclude)))
    print(f"    • {len(nodes_to_exclude)} noeuds sensibles identifiés pour exclusion (Mixed Precision) :")
    for i, name in enumerate(nodes_to_exclude):
        print(f"       [{i+1:02d}] {name}")
    return nodes_to_exclude

In [7]:

def quantize_model():
    print("=" * 70)
    print("  QUANTIFICATION YOLOP POUR DÉPLOIEMENT NPU (INT8)")
    print("=" * 70)
    
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    if not os.path.exists(INPUT_MODEL_PATH):
        print(f" ERREUR : Modèle introuvable : {INPUT_MODEL_PATH}")
        return
    
    # ============================================================
    # ÉTAPE 1 : SIMPLIFICATION (Fusion Conv+BN)
    # ============================================================
    print("\n[1/4] Simplification du modèle (fusion Conv+BN)...")
    SIMPLIFIED_PATH = os.path.join(OUTPUT_DIR, "temp_simplified.onnx")
    
    model_proto = onnx.load(INPUT_MODEL_PATH)
    model_sim, check = simplify(model_proto)
    
    if not check:
        print(" Échec de la simplification")
        return
    
    onnx.save(model_sim, SIMPLIFIED_PATH)
    print(f" Modèle simplifié : {SIMPLIFIED_PATH}")

    print(f"    Validation et pré-traitement ONNX Runtime...")
    PREPROCESSED_PATH = os.path.join(OUTPUT_DIR, "temp_preprocessed.onnx")
    quant_pre_process(SIMPLIFIED_PATH, PREPROCESSED_PATH)
    
    if os.path.exists(SIMPLIFIED_PATH):
        os.remove(SIMPLIFIED_PATH)
        
    SIMPLIFIED_PATH = PREPROCESSED_PATH
    print(f" Modèle pré-traité : {SIMPLIFIED_PATH}")
    
    # ============================================================
    # ÉTAPE 2 : IDENTIFICATION DES POIDS BATCHNORM
    # ============================================================
    print("\n[2/4] Identification des poids BatchNorm...")
    bn_weights = get_bn_weight_names(SIMPLIFIED_PATH)
    
    sensitive_nodes = get_sensitive_nodes(SIMPLIFIED_PATH, output_names=HEADS_TO_EXCLUDE, depth=EXCLUSION_DEPTH, manual_list=MANUAL_NODES_TO_EXCLUDE, force_int8_list=NODES_TO_FORCE_INT8)
    # ============================================================
    # ÉTAPE 3 : CALIBRATION
    # ============================================================
    print(f"\n[3/4] Préparation de la calibration...")
    print(f"    Source : {CALIBRATION_IMG_DIR}")
    
    if not os.path.exists(CALIBRATION_IMG_DIR):
        print(f" ERREUR : Dossier de calibration introuvable")
        return
    
    dr = YOLOPDataReader(CALIBRATION_IMG_DIR, SIMPLIFIED_PATH, size_limit=CALIBRATION_SIZE, priority_list=PRIORITY_IMAGES)
    
    # ============================================================
    # ÉTAPE 4 : QUANTIFICATION AVEC PER_CHANNEL=FALSE
    # ============================================================
    print(f"\n[4/4] Quantification INT8 pour ...")
    print(f"    Format       : QDQ ")
    print(f"    Activations  : QUInt8")
    print(f"    Weights      : QInt8")
    print(f"    Per-channel  : Activé ")
    print(f"    Reduce range : Activé")
    
    try:
        quantize_static(
            model_input=SIMPLIFIED_PATH,
            model_output=OUTPUT_MODEL_PATH,
            calibration_data_reader=dr,
            quant_format=QuantFormat.QDQ,         
            activation_type=QuantType.QUInt8,      
            weight_type=QuantType.QInt8,          
            per_channel=True,                    
            reduce_range=True,
            calibrate_method=CalibrationMethod.Entropy,
            nodes_to_exclude=sensitive_nodes                   
        )
        
        print("\n" + "=" * 70)
        print("  RÉSULTATS DE LA QUANTIFICATION")
        print("=" * 70)
        
        if os.path.exists(OUTPUT_MODEL_PATH):
            size_original = os.path.getsize(INPUT_MODEL_PATH) / (1024 * 1024)
            size_quantized = os.path.getsize(OUTPUT_MODEL_PATH) / (1024 * 1024)
            reduction = size_original / size_quantized
            
            print(f"\n Taille des modèles :")
            print(f"    Original (FP32)  : {size_original:>8.2f} MB")
            print(f"    Quantized (INT8) : {size_quantized:>8.2f} MB")
            print(f"    Réduction        : {reduction:>8.2f}x")
            
            print(f"\n Modèle INT8 :")
            print(f"    {OUTPUT_MODEL_PATH}")
            
            print(f"\n Caractéristiques du modèle :")
            print(f"    • Format : ONNX QDQ (Quantize-DeQuantize)")
            print(f"    • Précision : INT8")
            print(f"    • Optimisations : Conv+BN fusionnés")
            
            print(f"\n Nettoyage des fichiers temporaires...")
            if os.path.exists(SIMPLIFIED_PATH):
                os.remove(SIMPLIFIED_PATH)
                print(f"     temp_simplified.onnx supprimé")
            
            print("\n" + "=" * 70)

        else:
            print(" Erreur : Le fichier de sortie n'a pas été créé")
            
    except Exception as e:
        print(f"\n ERREUR lors de la quantification :")
        print(f"    {str(e)}")
        import traceback
        traceback.print_exc()



In [8]:

if __name__ == "__main__":
    quantize_model()

  QUANTIFICATION YOLOP POUR DÉPLOIEMENT NPU (INT8)

[1/4] Simplification du modèle (fusion Conv+BN)...
 Modèle simplifié : /workspace/weights/temp_simplified.onnx
    Validation et pré-traitement ONNX Runtime...
 Modèle pré-traité : /workspace/weights/temp_preprocessed.onnx

[2/4] Identification des poids BatchNorm...
 24 poids BatchNorm identifiés
    • Têtes identifiées pour exclusion : ['Concat_1534']
    • Forçage INT8 sur 33 noeuds (Retrait de la liste sensible)...
      - 31 noeuds réintégrés à la quantification.
    • 0 noeuds sensibles identifiés pour exclusion (Mixed Precision) :

[3/4] Préparation de la calibration...
    Source : /workspace/dataset/images/val
 DataReader : 50 images de calibration (Dont 1 prioritaires)

[4/4] Quantification INT8 pour ...
    Format       : QDQ 
    Activations  : QUInt8
    Weights      : QInt8
    Per-channel  : Activé 
    Reduce range : Activé
Finding optimal threshold for each tensor using 'entropy' algorithm ...
Number of tensors : 524



  RÉSULTATS DE LA QUANTIFICATION

 Taille des modèles :
    Original (FP32)  :    34.24 MB
    Quantized (INT8) :    11.83 MB
    Réduction        :     2.89x

 Modèle INT8 :
    /workspace/weights/yolop-640-640-quantized-int8-Entropy-per_channel-full.onnx

 Caractéristiques du modèle :
    • Format : ONNX QDQ (Quantize-DeQuantize)
    • Précision : INT8
    • Optimisations : Conv+BN fusionnés

 Nettoyage des fichiers temporaires...
     temp_simplified.onnx supprimé

